# Document-Level Section Propagation — The Headline Contribution

medspaCy operates sentence-by-sentence. cwyde's `section_propagator` component reads the 
document's section structure (produced by medspaCy's sectionizer) and applies section-level 
assertion context to every entity in that section.

This notebook uses a realistic CTPA (CT Pulmonary Angiography) report to demonstrate why 
sentence-level processing alone is insufficient.

In [1]:
import cwyde
from medspacy.target_matcher import TargetRule

nlp = cwyde.load("en")
matcher = nlp.get_pipe("medspacy_target_matcher")
matcher.add([
    TargetRule("PE", "CONDITION"),
    TargetRule("pulmonary embolism", "CONDITION"),
    TargetRule("DVT", "CONDITION"),
    TargetRule("pleural effusion", "CONDITION"),
    TargetRule("subsegmental PE", "CONDITION"),
    TargetRule("chest pain", "SYMPTOM"),
])

In [ ]:
report = """INDICATION: Chest pain and elevated D-dimer. Rule out pulmonary embolism.

PAST MEDICAL HISTORY: DVT two years ago. No prior PE.

INTERPRETATION: No filling defect to suggest pulmonary embolism. No pleural effusion.

IMPRESSION: No evidence of pulmonary embolism. Cannot exclude subsegmental PE."""

doc = nlp(report)

## Entity-by-entity walkthrough

For each matched entity we show:
- the sentence it appears in
- its `cwyde_assertion_category`
- whether `cwyde_section_inherited` is True (section propagator changed the category)
- the resolution trace

Expected highlights:
- `chest pain` in INDICATION section → `INDICATION`, `section_inherited=True` ← **headline**: no trigger word in the sentence; section context provides the classification
- `pulmonary embolism` in INDICATION section → `INDICATION`, `section_inherited=False` (indication_detector caught "rule out" first)
- `DVT` in PAST MEDICAL HISTORY → `HISTORICAL` (ConText fires "PAST MEDICAL HISTORY" as a trigger)
- `PE` in "No prior PE." → `DEFINITE_NEGATED_EXISTENCE` (direct negation; "no" forward trigger, sentence-local)
- `pulmonary embolism` in "No filling defect to suggest PE." → `PROBABLE_NEGATED_EXISTENCE` ← **richer taxonomy**: "to suggest" is a hedge; absence of evidence, not direct assertion of absence
- `pleural effusion` in "No pleural effusion." → `DEFINITE_NEGATED_EXISTENCE` (direct negation)
- `pulmonary embolism` in "No evidence of pulmonary embolism." → `DEFINITE_NEGATED_EXISTENCE`
- `subsegmental PE` in "Cannot exclude subsegmental PE." → `AMBIVALENT_EXISTENCE`

In [3]:
print(f"{'Entity':<22} {'Category':<30} {'Inherited':<10} {'Trace summary'}")
print("-" * 90)
for ent in doc.ents:
    category = ent._.cwyde_assertion_category.value
    inherited = str(ent._.cwyde_section_inherited)
    trace = ent._.cwyde_resolution_trace or []
    # Condense the trace to first step for display
    trace_summary = str(trace[0]) if trace else "(no conflict)"
    if len(trace_summary) > 40:
        trace_summary = trace_summary[:37] + "..."
    print(f"{ent.text:<22} {category:<30} {inherited:<10} {trace_summary}")

Entity                 Category                       Inherited  Trace summary
------------------------------------------------------------------------------------------
Chest pain             INDICATION                     True       {'step': 'category_mapper', 'medspacy...
pulmonary embolism     INDICATION                     False      {'step': 'category_mapper', 'medspacy...
DVT                    HISTORICAL                     False      {'step': 'category_mapper', 'medspacy...
PE                     DEFINITE_NEGATED_EXISTENCE     False      {'step': 'category_mapper', 'medspacy...
pulmonary embolism     DEFINITE_NEGATED_EXISTENCE     False      {'step': 'category_mapper', 'medspacy...
pleural effusion       DEFINITE_NEGATED_EXISTENCE     False      {'step': 'category_mapper', 'medspacy...
pulmonary embolism     DEFINITE_NEGATED_EXISTENCE     False      {'step': 'category_mapper', 'medspacy...
subsegmental PE        AMBIVALENT_EXISTENCE           False      {'step': 'category_mapp

## Key observations

1. **Section propagation fills in what sentence-level processing cannot.** `chest pain` in `INDICATION: Chest pain and elevated D-dimer.` has no indication trigger word. At sentence level it gets `DEFINITE_EXISTENCE`. The `section_propagator` promotes it to `INDICATION`. `cwyde_section_inherited=True` marks this as section-derived.

2. **Indication detector and section propagator are complementary, not redundant.** `pulmonary embolism` in the same INDICATION sentence already has `INDICATION` from the indication_detector (which caught "rule out"). Section propagation is a no-op; `section_inherited` stays `False`.

3. **HISTORICAL section does not override confirmed negation.** `No prior PE` in PAST MEDICAL HISTORY: sentence-level "no" gives `DEFINITE_NEGATED_EXISTENCE`. `override_existing: false` means the section assertion yields to the sentence-level result.

4. **The DEFINITE/PROBABLE distinction captures hedged radiology language.** "No filling defect to suggest pulmonary embolism" → `PROBABLE_NEGATED_EXISTENCE`. "No pulmonary embolism" → `DEFINITE_NEGATED_EXISTENCE`. The phrase "to suggest" is an epistemic hedge — the radiologist is reporting absence of an inferential sign, not directly asserting absence of the diagnosis. pyConTextNLP collapsed both to NEGATED; cwyde's richer taxonomy expresses the difference. The longer regex match `(?i)\bno\b[^\n.!?]+\bto suggest\b` wins medspaCy's overlap pruning over the shorter plain `no` trigger.

5. **Unmapped sections get only sentence-level processing.** INTERPRETATION (medspaCy `imaging`) and IMPRESSION (`observation_and_plan`) have no entry in `section_assertions.yaml`. Entities in those sections receive exactly what ConText provides.

6. **`cwyde_section_inherited` is the audit flag.** Downstream systems can distinguish section-propagated inference (`True`) from direct sentence-level evidence (`False`), supporting explainability and confidence tiering.

In [4]:
# Section-level assertions registered on the doc
section_assertions = doc._.cwyde_section_assertions
print("Document section assertions:")
if section_assertions:
    for section, assertion in section_assertions.items():
        print(f"  {section}: {assertion}")
else:
    print("  (none recorded — check that medspacy_sectionizer loaded section rules)")

Document section assertions:
  labs_and_studies: AssertionCategory.INDICATION
  past_medical_history: AssertionCategory.HISTORICAL
